# ECS7001P Assignment 2: Multimodal Question Answering

**Project Overview: Building a Custom LLaVA Architecture**

In this assignment, you will tackle a **Multimodal Question Answering (MQA)** task using a proprietary "in-house" corpus. This dataset consists of images paired with multiple-choice questions (MCQs). Your objective is to architect and train a customized **LLaVA (Large Language-and-Vision Assistant)** model by integrating a specialized vision encoder with a lightweight Large Language Model (LLM).



In [ ]:
!pip install transformers

**Dataset Description**

The dataset consists of **11,600 unique images**, each paired with 5–6 MCQs, totaling approximately **65k questions**.

**Data Splits**

* **Training Set:** 10k images and 56k MCQs. (Alignment is already provided).
* **Validation Set:** 600 images and 3.3k MCQs.
* **Test Set:** 1k images and 5.5k MCQs.

**File Structure**
The data is organized into an image repository and two types of `.jsonl` metadata files:

* **images/train/**: Training image set.
* **images/test_val/**: Combined images for both validation and testing.
* **vqa_xxx_student.jsonl**: Contains the MCQs, multiple-choice options, and metadata.
* **caption_xxx_student.jsonl**: Contains the image captions used to map images back to their respective questions.

**Note:** There is no caption file for the **Training** set, as the image-to-MCQ alignment is already provided for your supervised learning phase. Further **vqa_val_teacher.jsonl** were provided for you to evaluate the system's performace in the development phrase.

**Data:** the dataset (~11.6K images, ~65K multiple-choice questions) was provided by the module and is not redistributed here. Place the `images/` folder and the `.jsonl` files next to this notebook to run it.

In [ ]:
import torch, os, json
from transformers import CLIPModel, CLIPProcessor,CLIPVisionConfig
from transformers.image_utils import load_image

**Technical Specifications**
You will be working with the following components:

*    **Vision Encoder:** AlpachinoNLP/LongCLIP-ViT-B-32 (Optimized for long-form captioning and extended visual context).

*    **Language Backbone:** Qwen3-0.6b (A high-efficiency, small-scale LLM).

*    **The Goal:** Align visual features with linguistic embeddings to enable the model to "reason" over images to answer complex MCQs.
**Note:** You are **NOT** allowed to use any other pretrained models to complete this assingment.

In [ ]:
CLIP_MODEL_ID = "AlpachinoNLP/LongCLIP-ViT-B-32"
TEXI_MODEL_ID = "Qwen/Qwen3-0.6B"

**Task 1: Image-MCQ Reconstruction**

Before training the LLaVA model, you must reconstruct the image-MCQ mappings for the validation and test sets. Since the `test_val` folder provides images in bulk, you will use the **LongCLIP** vision head and the provided captions to perform semantic alignment.

**Objectives:**
* Implement the `recover_image_id()` method to perform semantic alignment between image features and captions.
* Utilize the `caption_ids` key present in the relevant `.jsonl` files to link captions and MCQs.
* Recover the `image` field in `vqa_val_student.jsonl` and `vqa_test_student.jsonl`. This field must store the relative path (e.g., `"images/test_val/000957012.jpg"`).
* Save the resulting files as `vqa_val_student_with_image_ids.jsonl` and `vqa_test_student_with_image_ids.jsonl`.

**Evaluation:**

The provided **LongCLIP** model is pre-trained on an extended version of this corpus and is highly accurate. A successful implementation should achieve **>99% accuracy** when evaluated using the provided `evaluate_image_caption_alignment()` method.

*Note: Feel free to add helper methods, ensuring they are clearly commented for clarity.*

In [ ]:
def evaluate_image_caption_alignment(gold_path,pred_path):
  gold_data = set()
  pred_data = set()
  for line in open(gold_path, 'r'):
    jdata = json.loads(line)
    gold_data.add((jdata['caption_id'],jdata['image']))
  for line in open(pred_path, 'r'):
    jdata = json.loads(line)
    pred_data.add((jdata['caption_id'],jdata['image']))
  return len(gold_data & pred_data)/len(gold_data)

In [ ]:
def recover_image_id():
    """
    Task 1: Recover image paths for validation and test VQA files.

    The idea is:
    1. Load LongCLIP.
    2. Encode all images from images/test_val/.
    3. Encode all captions from caption_val_student.jsonl and caption_test_student.jsonl.
    4. Match each caption to the most similar image using cosine similarity.
    5. Add the recovered image path to each VQA item using caption_ids.
    6. Save the new VQA files with image ids.
    """

    import glob
    from tqdm import tqdm
    from PIL import Image

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Using device:", device)

    # Load LongCLIP
    clip_model = CLIPModel.from_pretrained(CLIP_MODEL_ID).to(device)
    clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
    clip_model.eval()

    # Collect all images in the validation/test image folder
    image_paths = sorted(glob.glob("images/test_val/*.jpg"))
    image_paths = [p.replace("\\", "/") for p in image_paths]

    print("Number of candidate images:", len(image_paths))

    def read_jsonl(path):
        """Read a jsonl file into a list of dictionaries."""
        data = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    data.append(json.loads(line))
        return data

    def write_jsonl(data, path):
        """Write a list of dictionaries to a jsonl file."""
        with open(path, "w", encoding="utf-8") as f:
            for item in data:
                f.write(json.dumps(item) + "\n")

    def get_caption_text(item):
        possible_keys = ["caption", "text", "sentence", "description"]

        for key in possible_keys:
            if key in item:
                return item[key]

        raise KeyError(f"Could not find caption text key. Available keys: {list(item.keys())}")

    def get_caption_id(item):
        if "caption_id" in item:
            return item["caption_id"]

        if "id" in item:
            return item["id"]

        raise KeyError(f"Could not find caption id key. Available keys: {list(item.keys())}")

    def encode_images(paths, batch_size=32):
        """
        Encode image files using LongCLIP image encoder.
        Features are normalized so that dot product gives cosine similarity.
        """
        all_features = []

        with torch.no_grad():
            for i in tqdm(range(0, len(paths), batch_size), desc="Encoding images"):
                batch_paths = paths[i:i + batch_size]

                images = []
                for path in batch_paths:
                    image = Image.open(path).convert("RGB")
                    images.append(image)

                inputs = clip_processor(
                    images=images,
                    return_tensors="pt",
                    padding=True
                ).to(device)

                outputs = clip_model.vision_model(
                    pixel_values=inputs["pixel_values"]
                )

                # LongCLIP returns a model output object so we use pooler_output
                features = outputs.pooler_output

                # Apply the visual projection if it exists
                if hasattr(clip_model, "visual_projection"):
                    features = clip_model.visual_projection(features)

                features = features / features.norm(dim=-1, keepdim=True)

                all_features.append(features.cpu())

        return torch.cat(all_features, dim=0)

    def encode_captions(captions, batch_size=64):
        """
        Encode caption texts using LongCLIP text encoder.
        """
        all_features = []

        with torch.no_grad():
            for i in tqdm(range(0, len(captions), batch_size), desc="Encoding captions"):
                batch_captions = captions[i:i + batch_size]

                inputs = clip_processor(
                    text=batch_captions,
                    return_tensors="pt",
                    padding=True,
                    truncation=True
                ).to(device)

                outputs = clip_model.text_model(
                    input_ids=inputs["input_ids"],
                    attention_mask=inputs["attention_mask"]
                )

                # LongCLIP returns a model output object so we use pooler_output
                features = outputs.pooler_output

                # Apply the text projection if it exists
                if hasattr(clip_model, "text_projection"):
                    features = clip_model.text_projection(features)

                features = features / features.norm(dim=-1, keepdim=True)

                all_features.append(features.cpu())

        return torch.cat(all_features, dim=0)

    def recover_one_split(caption_path, vqa_path, output_path, image_features):
        """
        Recover image paths for one split: validation or test.
        """
        caption_data = read_jsonl(caption_path)
        vqa_data = read_jsonl(vqa_path)

        caption_ids = []
        caption_texts = []

        for item in caption_data:
            caption_ids.append(get_caption_id(item))
            caption_texts.append(get_caption_text(item))

        print(f"\nRecovering image ids for: {vqa_path}")
        print("Number of captions:", len(caption_texts))

        # Encode captions
        caption_features = encode_captions(caption_texts)

        # Compare caption features with all image features
        # Shape: number_of_captions x number_of_images
        similarity = caption_features @ image_features.T

        # Pick the most similar image for each caption
        best_image_indices = similarity.argmax(dim=1).tolist()

        # Create caption_id -> image_path mapping
        caption_to_image = {}

        for caption_id, image_index in zip(caption_ids, best_image_indices):
            caption_to_image[caption_id] = image_paths[image_index]

        recovered_data = []

        for item in vqa_data:
            # VQA files use caption_ids, but evaluation expects caption_id.
            if "caption_id" in item:
                caption_id = item["caption_id"]
            elif "caption_ids" in item:
                caption_id = item["caption_ids"][0]
                item["caption_id"] = caption_id
            else:
                raise KeyError(f"No caption_id or caption_ids found. Available keys: {list(item.keys())}")

            item["image"] = caption_to_image[caption_id]
            recovered_data.append(item)

        write_jsonl(recovered_data, output_path)
        print("Saved:", output_path)

    # Encode all candidate images only once
    image_features = encode_images(image_paths)

    # Recover validation split
    recover_one_split(
        caption_path="caption_val_student.jsonl",
        vqa_path="vqa_val_student.jsonl",
        output_path="vqa_val_student_with_image_ids.jsonl",
        image_features=image_features
    )

    # Recover test split
    recover_one_split(
        caption_path="caption_test_student.jsonl",
        vqa_path="vqa_test_student.jsonl",
        output_path="vqa_test_student_with_image_ids.jsonl",
        image_features=image_features
    )

    # Evaluate validation alignment
    acc = evaluate_image_caption_alignment(
        "vqa_val_teacher.jsonl",
        "vqa_val_student_with_image_ids.jsonl"
    )

    print(f"\nEvaluation accuracy: {acc*100.0:.2f}%")

In [ ]:
recover_image_id()

Using device: cuda


Number of candidate images: 1600


Recovering image ids for: vqa_val_student.jsonl
Number of captions: 600


Saved: vqa_val_student_with_image_ids.jsonl
Recovering image ids for: vqa_test_student.jsonl
Number of captions: 1000


Saved: vqa_test_student_with_image_ids.jsonl
Evaluation accuracy: 99.83%


### Task 2: LLaVA Training & Evaluation
Once the mappings are restored, you will implement the LLaVA framework via the following steps:

1. **Model Construction:** Extract the vision head from **LongCLIP** and integrate it with **Qwen** to build your custom LLaVA model within the `train_evaluate_LLaVA_model()` method.
2. **Alignment:** Implement the `LlavaMultiModalProjector` class to map CLIP visual tokens into the Qwen3 embedding space using a projection layer (Linear or MLP).
3. **Fine-tuning:** Train the system to process an image and a text-based MCQ simultaneously to predict the correct answer using the `train_evaluate_LLaVA_model()` method.
4. **Inference:** Evaluate performance on the reconstructed validation and test pipelines using the `get_prediction()` and `evaluation()` methods.
5. **Output:** Export your final predictions into JSONL files named `vqa_val_student_final.jsonl` and `vqa_test_student_final.jsonl`.

In [ ]:
import random,json,datetime
import torch
import numpy as np
from PIL import Image
from torch import nn
from transformers import (AutoTokenizer,
                          CLIPImageProcessor,
                          AutoModelForCausalLM,
                          CLIPVisionModel,
                          LlavaForConditionalGeneration,
                          LlavaProcessor,
                          CLIPVisionConfig,
                          LlavaConfig)
from transformers.activations import ACT2FN

In [ ]:
# evaluate the vqa accuracy between the gold and predicted data
def evaluation(gold_path,pred_path):
  gold_data = []
  pred_data = []
  for line in open(gold_path, 'r'):
    jdata = json.loads(line)
    gold_data.append(jdata['answer'])
  for line in open(pred_path, 'r'):
    jdata = json.loads(line)
    pred_data.append(jdata['answer'])
  return sum(1 for i in range(len(gold_data)) if gold_data[i] == pred_data[i])/len(gold_data)


# get the predictions from the model for evaluation
def get_prediction(model, processor, batch_size, device, dataset):
    """
    Logit-based MCQ inference: no text generation, just compare
    the logits of A/B/C/D at the last token position and pick the best.
    Returns a list of predicted answer letters ['A', 'B', ...].
    """
    model.eval()
    predictions = []

    image_paths  = dataset['image_paths']
    prompts      = dataset['prompts']
    answer_letters = ['A', 'B', 'C', 'D']

    # Get the single token ID for each answer letter
    answer_token_ids = [
        processor.tokenizer.encode(letter, add_special_tokens=False)[0]
        for letter in answer_letters
    ]

    with torch.no_grad():
        for i in range(0, len(prompts), batch_size):
            batch_prompts      = prompts[i:i+batch_size]
            batch_image_paths  = image_paths[i:i+batch_size]

            images = [Image.open(p).convert('RGB') for p in batch_image_paths]

            inputs = processor(text=batch_prompts, images=images, return_tensors="pt", padding=True)
            inputs = {k: v.to(device) for k, v in inputs.items()}

            outputs   = model(**inputs)
            logits    = outputs.logits        # [batch, seq_len, vocab_size]

            # With left-padding the last column is always the last real token,
            # the position whose next-token distribution we want.
            last_logits     = logits[:, -1, :]                # [batch, vocab_size]
            answer_logits   = last_logits[:, answer_token_ids]  # [batch, 4]
            predicted_idx   = answer_logits.argmax(dim=-1)

            for idx in predicted_idx:
                predictions.append(answer_letters[idx.item()])

    return predictions

# output the predictions to the files
def write_answers_to_file(path, predictions, answer_to_ids, tag=""):
    """
    Writes predictions to a JSONL output file.
    tag parameter allows saving each experiment's results separately.
    """
    suffix = f"_{tag}" if tag else ""
    output_path = path.replace('with_image_ids', f'final{suffix}')
    ids_to_answer = {v: k for k, v in answer_to_ids.items()}

    with open(output_path, 'w', encoding='utf-8') as f_out:
        for line, pred in zip(open(path, 'r', encoding='utf-8'), predictions):
            jdata = json.loads(line)
            jdata['answer'] = ids_to_answer[pred] if isinstance(pred, int) else pred
            f_out.write(json.dumps(jdata) + '\n')

    print(f"Saved: {output_path}")


In [ ]:
# The projection layer for LLaVA
class LlavaMultiModalProjector(nn.Module):
    """
    Projects CLIP visual features into Qwen3's embedding space.
    - Input:  [batch, num_image_tokens, vision_hidden_size]
    - Output: [batch, num_image_tokens, text_hidden_size]
    """
    def __init__(self, config: LlavaConfig):
        super().__init__()
        self.linear_1 = nn.Linear(
            config.vision_config.hidden_size,
            config.text_config.hidden_size,
            bias=True
        )
        self.act = ACT2FN[config.projector_hidden_act]  # gelu by default
        self.linear_2 = nn.Linear(
            config.text_config.hidden_size,
            config.text_config.hidden_size,
            bias=True
        )

    def forward(self, image_features):
        hidden_states = self.linear_1(image_features)
        hidden_states = self.act(hidden_states)
        hidden_states = self.linear_2(hidden_states)
        return hidden_states


class SimpleProcessor:
    def __init__(self, tokenizer, image_processor):
        self.tokenizer       = tokenizer
        self.image_processor = image_processor

    def __call__(self, text, images, return_tensors="pt", padding=True):
        imgs = self.image_processor(images=images, return_tensors=return_tensors)
        txts = self.tokenizer(text, return_tensors=return_tensors,
                              padding=padding, truncation=True, max_length=512)
        return {**txts, "pixel_values": imgs.pixel_values}


class CustomLLaVA(nn.Module):
    def __init__(self, vision_model, projector, language_model, image_token_id):
        super().__init__()
        self.vision_model   = vision_model
        self.projector      = projector
        self.language_model = language_model
        self.image_token_id = image_token_id

    def _merge_image_text(self, input_ids, attention_mask, pixel_values):
        B = input_ids.shape[0]
        vis_out   = self.vision_model(pixel_values)
        img_feats = self.projector(vis_out.last_hidden_state)  # [B, P, H]
        P         = img_feats.shape[1]
        text_emb  = self.language_model.get_input_embeddings()(input_ids)

        embs, masks = [], []
        for b in range(B):
            pos = (input_ids[b] == self.image_token_id).nonzero(as_tuple=True)[0]
            if len(pos) == 0:
                embs.append(text_emb[b]); masks.append(attention_mask[b]); continue
            p = pos[0].item()
            embs.append(torch.cat([text_emb[b,:p], img_feats[b], text_emb[b,p+1:]], dim=0))
            masks.append(torch.cat([
                attention_mask[b,:p],
                torch.ones(P, dtype=attention_mask.dtype, device=attention_mask.device),
                attention_mask[b,p+1:]
            ], dim=0))

        max_len = max(e.shape[0] for e in embs)
        H = text_emb.shape[-1]
        pe = text_emb.new_zeros(B, max_len, H)
        pm = attention_mask.new_zeros(B, max_len)
        for b, (e, m) in enumerate(zip(embs, masks)):
            L = e.shape[0]; pe[b, max_len-L:] = e; pm[b, max_len-L:] = m
        return pe, pm

    def forward(self, input_ids, attention_mask, pixel_values):
        emb, mask = self._merge_image_text(input_ids, attention_mask, pixel_values)
        return self.language_model(inputs_embeds=emb, attention_mask=mask)


In [ ]:
# load the image paths and create prompt from MCQ
def load_image_vqa_data(path,answer_to_ids):
  image_paths = []
  prompts = []
  answers = []
  for line in open(path, 'r'):
    jdata = json.loads(line)
    image_paths.append(jdata['image'])
    question = jdata['question']
    options = jdata['options']
    answer = answer_to_ids.get(jdata['answer'],None)

    prompts.append("<image>\nAnswer with A, B, C, or D only.\n"
        f"Question: {question}\n"
        "Options:\n"
        f"A. {options[0]}\n"
        f"B. {options[1]}\n"
        f"C. {options[2]}\n"
        f"D. {options[3]}\n"
        "Answer:\n")
    answers.append(answer)
  return {"image_paths":image_paths, "prompts":prompts, "answers":answers}



def train_evaluate_LLaVA_model():
    random.seed(0)
    torch.manual_seed(0)

    device         = "cuda" if torch.cuda.is_available() else "cpu"
    answer_to_ids  = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
    answer_letters = ['A', 'B', 'C', 'D']

    # ── Task 2.1: tokenizer ─────
    from transformers import AutoConfig
    tokenizer = AutoTokenizer.from_pretrained(TEXI_MODEL_ID)
    tokenizer.add_special_tokens({"additional_special_tokens": ["<image>"]})
    image_token_id      = tokenizer.convert_tokens_to_ids("<image>")
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    print("Loading LongCLIP vision encoder...")
    clip_full     = CLIPModel.from_pretrained(CLIP_MODEL_ID)
    vision_model  = clip_full.vision_model      # vision tower only
    vision_config = vision_model.config
    del clip_full; torch.cuda.empty_cache()

    print("Loading Qwen3 language model...")
    qwen_model  = AutoModelForCausalLM.from_pretrained(TEXI_MODEL_ID)
    text_config = qwen_model.config
    qwen_model.resize_token_embeddings(len(tokenizer))   # make room for the <image> token

    # projector config
    import types
    proj_cfg = types.SimpleNamespace(
        vision_config        = types.SimpleNamespace(hidden_size=vision_model.config.hidden_size),
        text_config          = types.SimpleNamespace(hidden_size=qwen_model.config.hidden_size),
        projector_hidden_act = "gelu"
    )
    projector = LlavaMultiModalProjector(proj_cfg)

    # CustomLLaVA: LongCLIP + projector + Qwen3
    model = CustomLLaVA(vision_model, projector, qwen_model, image_token_id).to(device)

    for param in model.vision_model.parameters():
        param.requires_grad = False

    print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    # processor
    image_processor = CLIPImageProcessor.from_pretrained(CLIP_MODEL_ID)
    processor       = SimpleProcessor(tokenizer, image_processor)

    train_data = load_image_vqa_data('vqa_train_student.jsonl',                   answer_to_ids)
    val_data   = load_image_vqa_data('vqa_val_student_with_image_ids.jsonl',  answer_to_ids)
    test_data  = load_image_vqa_data('vqa_test_student_with_image_ids.jsonl', answer_to_ids)
    print(f"Train {len(train_data['prompts'])} | Val {len(val_data['prompts'])} | Test {len(test_data['prompts'])}")

    batch_size    = 8
    num_epochs    = 3
    learning_rate = 1e-4

    answer_token_ids = [tokenizer.encode(l, add_special_tokens=False)[0] for l in answer_letters]

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=learning_rate, weight_decay=0.01
    )
    total_steps = num_epochs * (len(train_data['prompts']) // batch_size)
    scheduler   = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-6)

    for epoch in range(num_epochs):
        model.train()
        total_loss, num_batches = 0.0, 0
        indices = list(range(len(train_data['prompts'])))
        random.shuffle(indices)

        for i in range(0, len(indices), batch_size):
            idx         = indices[i:i+batch_size]
            b_prompts   = [train_data['prompts'][j]     for j in idx]
            b_img_paths = [train_data['image_paths'][j] for j in idx]
            b_answers   = [train_data['answers'][j]     for j in idx]

            valid = [(p,ip,a) for p,ip,a in zip(b_prompts,b_img_paths,b_answers) if a is not None]
            if not valid: continue
            b_prompts, b_img_paths, b_answers = zip(*valid)

            try:
                b_images = [Image.open(p).convert('RGB') for p in b_img_paths]
            except Exception:
                continue

            inputs = processor(text=list(b_prompts), images=b_images, return_tensors="pt", padding=True)
            inputs = {k: v.to(device) for k, v in inputs.items()}

            outputs     = model(inputs['input_ids'], inputs['attention_mask'], inputs['pixel_values'])
            last_logits = outputs.logits[:, -1, :]
            targets     = torch.tensor([answer_token_ids[a] for a in b_answers], device=device)
            loss        = torch.nn.functional.cross_entropy(last_logits, targets)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item(); num_batches += 1

        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {total_loss/max(num_batches,1):.4f}")

    final_val_predictions  = get_prediction(model, processor, batch_size, device, val_data)
    final_test_predictions = get_prediction(model, processor, batch_size, device, test_data)
    write_answers_to_file('vqa_val_student_with_image_ids.jsonl',  final_val_predictions,  answer_to_ids)
    write_answers_to_file('vqa_test_student_with_image_ids.jsonl', final_test_predictions, answer_to_ids)
    val_eval_acc = evaluation('vqa_val_teacher.jsonl', 'vqa_val_student_final.jsonl')
    print(f"Final evaluation accuracy on val set: {val_eval_acc*100.0:.2f}%")

torch.cuda.empty_cache()
train_evaluate_LLaVA_model()


Loading LongCLIP vision encoder...


Loading Qwen3 language model...


Trainable params: 597,614,592
Train 56055 | Val 3300 | Test 5538
Epoch 1/3 | Loss: 1.4344
Epoch 2/3 | Loss: 1.3017
Epoch 3/3 | Loss: 1.0987
Saved: vqa_val_student_final.jsonl
Saved: vqa_test_student_final.jsonl
Final evaluation accuracy on val set: 50.94%


**Task 3: Performance Optimization**

The final goal is to improve the model's reasoning capabilities and MCQ accuracy. You are encouraged to experiment with advanced training and architectural strategies.

**Potential Areas for Improvement:**
* **Architecture Tweaks:** Experiment with more advanced projectors beyond a basic MLP.
* **Fine-tuning Techniques:** Implement **LoRA (Low-Rank Adaptation)** or other PEFT methods to fine-tune the Qwen3 backbone more effectively.
* **Prompt Engineering:** Optimize the text instructions provided to the model (e.g., more detailed formatting or specific task-oriented prompting).
* **Data Augmentation:** Use image or text data augmentation techniques to enhance the diversity of the training data.
* **Hyperparameter Tuning:** Adjust learning rates, scheduler types, and batch sizes to stabilize convergence and improve results.

**Requirement:** Select **at least 3 categories** from the list above. For each chosen category, explain your intuition behind the changes, document your experiments, and provide a comparative analysis of how these modifications impacted your final accuracy on the **validation set**.

In [ ]:
import math

# Enhanced projector: 3-layer MLP with LayerNorm after each hidden layer.
# Baseline uses a plain 2-layer MLP with no normalisation.
class EnhancedProjector(nn.Module):
    def __init__(self, vision_hidden_size, text_hidden_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(vision_hidden_size, text_hidden_size),
            nn.LayerNorm(text_hidden_size), nn.GELU(),
            nn.Linear(text_hidden_size, text_hidden_size),
            nn.LayerNorm(text_hidden_size), nn.GELU(),
            nn.Linear(text_hidden_size, text_hidden_size),
        )

    def forward(self, x):
        # x: [batch, num_patches, vision_hidden]
        return self.net(x)


# Improved prompt template used in Experiment 2.
# Defined here so run_experiment() can reference it even in Exp 1.
def load_image_vqa_data_v2(path, answer_to_ids):
    """
    Variant of load_image_vqa_data with a more explicit prompt.
    Tells the model to inspect the image and uses a clearer answer cue.
    """
    image_paths, prompts, answers = [], [], []
    for line in open(path, 'r'):
        jdata  = json.loads(line)
        opts   = jdata['options']
        prompt = (
            "<image>\n"
            "Inspect the image carefully and answer the question. "
            "Reply with only A, B, C, or D.\n"
            f"Question: {jdata['question']}\n---\n"
            f"A. {opts[0]}\nB. {opts[1]}\nC. {opts[2]}\nD. {opts[3]}\n---\n"
            "The answer is:"
        )
        image_paths.append(jdata['image'])
        prompts.append(prompt)
        answers.append(answer_to_ids.get(jdata['answer'], None))
    return {"image_paths": image_paths, "prompts": prompts, "answers": answers}


# Warmup + cosine decay scheduler used in Experiment 3.
# Defined here so run_experiment() can reference it even in Exp 1.
def get_warmup_cosine_scheduler(optimizer, warmup_steps, total_steps):
    """
    Linear warmup for warmup_steps, then cosine decay to 1% of peak LR.
    Warmup prevents large updates from destabilising the random projector early on.
    """
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.01, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def build_base_model(tokenizer, image_token_id, use_enhanced_projector=False):
    """
    Loads LongCLIP vision encoder and Qwen3 LLM, then attaches the chosen projector.
    Shared across all three experiments to ensure a fair comparison.
    """
    clip_full    = CLIPModel.from_pretrained(CLIP_MODEL_ID)
    vision_model = clip_full.vision_model
    del clip_full; torch.cuda.empty_cache()

    qwen_model = AutoModelForCausalLM.from_pretrained(TEXI_MODEL_ID)
    qwen_model.resize_token_embeddings(len(tokenizer))

    vision_h = vision_model.config.hidden_size
    text_h   = qwen_model.config.hidden_size

    if use_enhanced_projector:
        projector = EnhancedProjector(vision_h, text_h)
    else:
        # Baseline 2-layer MLP projector
        import types
        cfg = types.SimpleNamespace(
            vision_config=types.SimpleNamespace(hidden_size=vision_h),
            text_config=types.SimpleNamespace(hidden_size=text_h),
            projector_hidden_act="gelu"
        )
        projector = LlavaMultiModalProjector(cfg)

    return vision_model, projector, qwen_model


def run_experiment(name, num_epochs, use_enhanced_projector, use_v2_prompt, use_warmup, peak_lr=1e-4):
    """
    Trains and evaluates one experiment configuration.
    Returns the validation accuracy so results can be compared against the baseline.
    """
    print(f"\n{'='*55}")
    print(f"EXPERIMENT: {name}")
    print(f"  enhanced_proj={use_enhanced_projector} | v2_prompt={use_v2_prompt} | warmup={use_warmup} | lr={peak_lr}")
    print(f"{'='*55}")

    random.seed(0); torch.manual_seed(0)
    device         = "cuda" if torch.cuda.is_available() else "cpu"
    answer_to_ids  = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
    answer_letters = ['A', 'B', 'C', 'D']

    # Tokenizer with custom <image> token
    tokenizer = AutoTokenizer.from_pretrained(TEXI_MODEL_ID)
    tokenizer.add_special_tokens({"additional_special_tokens": ["<image>"]})
    image_token_id      = tokenizer.convert_tokens_to_ids("<image>")
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    vision_model, projector, qwen_model = build_base_model(
        tokenizer, image_token_id, use_enhanced_projector
    )
    model = CustomLLaVA(vision_model, projector, qwen_model, image_token_id).to(device)

    # Freeze vision encoder for Task 1
    for p in model.vision_model.parameters():
        p.requires_grad = False

    print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    image_processor = CLIPImageProcessor.from_pretrained(CLIP_MODEL_ID)
    processor       = SimpleProcessor(tokenizer, image_processor)

    # Select prompt loader based on experiment config
    loader     = load_image_vqa_data_v2 if use_v2_prompt else load_image_vqa_data
    train_data = loader('vqa_train_student.jsonl',                 answer_to_ids)
    val_data   = loader('vqa_val_student_with_image_ids.jsonl',  answer_to_ids)
    test_data  = loader('vqa_test_student_with_image_ids.jsonl', answer_to_ids)
    print(f"Train {len(train_data['prompts'])} | Val {len(val_data['prompts'])} | Test {len(test_data['prompts'])}")

    batch_size       = 8
    answer_token_ids = [tokenizer.encode(l, add_special_tokens=False)[0] for l in answer_letters]

    optimizer   = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=peak_lr, weight_decay=0.01
    )
    total_steps = num_epochs * (len(train_data['prompts']) // batch_size)

    # Select LR scheduler based on experiment config
    if use_warmup:
        scheduler = get_warmup_cosine_scheduler(optimizer, int(total_steps * 0.05), total_steps)
    else:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-6)

    for epoch in range(num_epochs):
        model.train()
        total_loss, n = 0.0, 0
        indices = list(range(len(train_data['prompts']))); random.shuffle(indices)

        for i in range(0, len(indices), batch_size):
            idx  = indices[i:i+batch_size]
            b_p  = [train_data['prompts'][j]     for j in idx]
            b_ip = [train_data['image_paths'][j] for j in idx]
            b_a  = [train_data['answers'][j]     for j in idx]

            valid = [(p,ip,a) for p,ip,a in zip(b_p,b_ip,b_a) if a is not None]
            if not valid: continue
            b_p, b_ip, b_a = zip(*valid)

            try:
                b_img = [Image.open(p).convert('RGB') for p in b_ip]
            except:
                continue

            inp = processor(text=list(b_p), images=b_img, return_tensors="pt", padding=True)
            inp = {k: v.to(device) for k, v in inp.items()}

            out  = model(inp['input_ids'], inp['attention_mask'], inp['pixel_values'])
            tgt  = torch.tensor([answer_token_ids[a] for a in b_a], device=device)
            loss = torch.nn.functional.cross_entropy(out.logits[:, -1, :], tgt)

            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            total_loss += loss.item(); n += 1

        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {total_loss/max(n,1):.4f}")

    # Save predictions for both val and test sets
    val_preds  = get_prediction(model, processor, batch_size, device, val_data)
    test_preds = get_prediction(model, processor, batch_size, device, test_data)
    write_answers_to_file('vqa_val_student_with_image_ids.jsonl',  val_preds,  answer_to_ids)
    write_answers_to_file('vqa_test_student_with_image_ids.jsonl', test_preds, answer_to_ids)

    acc = evaluation('vqa_val_teacher.jsonl', 'vqa_val_student_final.jsonl')
    print(f"\n→ Val accuracy: {acc*100:.2f}%  (baseline: 50.94%)")

    del model; torch.cuda.empty_cache()
    return acc


# ── Experiment 1: Architecture Tweaks — Enhanced Projector only ──────
# Intuition: LayerNorm stabilises feature magnitudes between projection
# layers, preventing scale mismatch between CLIP and Qwen3 embeddings.
# A third linear layer increases the projector's mapping capacity.
torch.cuda.empty_cache()
acc1 = run_experiment(
    "Exp1 - Enhanced Projector only",
    num_epochs=3,
    use_enhanced_projector=True,
    use_v2_prompt=False,
    use_warmup=False,
    peak_lr=1e-4
)

EXPERIMENT: Exp1 - Enhanced Projector only
  enhanced_proj=True | v2_prompt=False | warmup=False | lr=0.0001


Trainable params: 598,668,288
Train 56055 | Val 3300 | Test 5538
Epoch 1/3 | Loss: 1.4244
Epoch 2/3 | Loss: 1.3231
Epoch 3/3 | Loss: 1.0061
Saved: vqa_val_student_final.jsonl
Saved: vqa_test_student_final.jsonl
→ Val accuracy: 54.36%  (baseline: 50.94%)


In [ ]:
import math

class EnhancedProjector(nn.Module):
    def __init__(self, vision_hidden_size, text_hidden_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(vision_hidden_size, text_hidden_size),
            nn.LayerNorm(text_hidden_size), nn.GELU(),
            nn.Linear(text_hidden_size, text_hidden_size),
            nn.LayerNorm(text_hidden_size), nn.GELU(),
            nn.Linear(text_hidden_size, text_hidden_size),
        )
    def forward(self, x):
        return self.net(x)


def load_image_vqa_data_v2(path, answer_to_ids):
    """
    Improved prompt variant: explicitly tells the model to inspect the image
    and uses a clearer answer cue ('The answer is:') instead of 'Answer:'.
    """
    image_paths, prompts, answers = [], [], []
    for line in open(path, 'r'):
        jdata  = json.loads(line)
        opts   = jdata['options']
        prompt = (
            "<image>\n"
            "Inspect the image carefully and answer the question. "
            "Reply with only A, B, C, or D.\n"
            f"Question: {jdata['question']}\n---\n"
            f"A. {opts[0]}\nB. {opts[1]}\nC. {opts[2]}\nD. {opts[3]}\n---\n"
            "The answer is:"
        )
        image_paths.append(jdata['image'])
        prompts.append(prompt)
        answers.append(answer_to_ids.get(jdata['answer'], None))
    return {"image_paths": image_paths, "prompts": prompts, "answers": answers}


def get_warmup_cosine_scheduler(optimizer, warmup_steps, total_steps):
    """
    Linear warmup for warmup_steps, then cosine decay to 1% of peak LR.
    """
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.01, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def build_base_model(tokenizer, image_token_id, use_enhanced_projector=False):
    """
    Loads LongCLIP vision encoder and Qwen3 LLM, then attaches the chosen projector.
    Shared across all experiments to ensure a fair comparison.
    """
    clip_full    = CLIPModel.from_pretrained(CLIP_MODEL_ID)
    vision_model = clip_full.vision_model
    del clip_full; torch.cuda.empty_cache()

    qwen_model = AutoModelForCausalLM.from_pretrained(TEXI_MODEL_ID)
    qwen_model.resize_token_embeddings(len(tokenizer))

    vision_h = vision_model.config.hidden_size
    text_h   = qwen_model.config.hidden_size

    if use_enhanced_projector:
        projector = EnhancedProjector(vision_h, text_h)
    else:
        import types
        cfg = types.SimpleNamespace(
            vision_config=types.SimpleNamespace(hidden_size=vision_h),
            text_config=types.SimpleNamespace(hidden_size=text_h),
            projector_hidden_act="gelu"
        )
        projector = LlavaMultiModalProjector(cfg)

    return vision_model, projector, qwen_model


def run_experiment(name, num_epochs, use_enhanced_projector, use_v2_prompt, use_warmup, peak_lr=1e-4):
    """
    Trains and evaluates one experiment configuration.
    Returns validation accuracy for comparison against the baseline.
    """
    print(f"\n{'='*55}")
    print(f"EXPERIMENT: {name}")
    print(f"  enhanced_proj={use_enhanced_projector} | v2_prompt={use_v2_prompt} | warmup={use_warmup} | lr={peak_lr}")
    print(f"{'='*55}")

    random.seed(0); torch.manual_seed(0)
    device         = "cuda" if torch.cuda.is_available() else "cpu"
    answer_to_ids  = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
    answer_letters = ['A', 'B', 'C', 'D']

    tokenizer = AutoTokenizer.from_pretrained(TEXI_MODEL_ID)
    tokenizer.add_special_tokens({"additional_special_tokens": ["<image>"]})
    image_token_id      = tokenizer.convert_tokens_to_ids("<image>")
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    vision_model, projector, qwen_model = build_base_model(
        tokenizer, image_token_id, use_enhanced_projector
    )
    model = CustomLLaVA(vision_model, projector, qwen_model, image_token_id).to(device)

    for p in model.vision_model.parameters():
        p.requires_grad = False

    print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    image_processor = CLIPImageProcessor.from_pretrained(CLIP_MODEL_ID)
    processor       = SimpleProcessor(tokenizer, image_processor)

    loader     = load_image_vqa_data_v2 if use_v2_prompt else load_image_vqa_data
    train_data = loader('vqa_train_student.jsonl',                 answer_to_ids)
    val_data   = loader('vqa_val_student_with_image_ids.jsonl',  answer_to_ids)
    test_data  = loader('vqa_test_student_with_image_ids.jsonl', answer_to_ids)
    print(f"Train {len(train_data['prompts'])} | Val {len(val_data['prompts'])} | Test {len(test_data['prompts'])}")

    batch_size       = 8
    answer_token_ids = [tokenizer.encode(l, add_special_tokens=False)[0] for l in answer_letters]

    optimizer   = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=peak_lr, weight_decay=0.01
    )
    total_steps = num_epochs * (len(train_data['prompts']) // batch_size)

    if use_warmup:
        scheduler = get_warmup_cosine_scheduler(optimizer, int(total_steps * 0.05), total_steps)
    else:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-6)

    for epoch in range(num_epochs):
        model.train()
        total_loss, n = 0.0, 0
        indices = list(range(len(train_data['prompts']))); random.shuffle(indices)

        for i in range(0, len(indices), batch_size):
            idx  = indices[i:i+batch_size]
            b_p  = [train_data['prompts'][j]     for j in idx]
            b_ip = [train_data['image_paths'][j] for j in idx]
            b_a  = [train_data['answers'][j]     for j in idx]

            valid = [(p,ip,a) for p,ip,a in zip(b_p,b_ip,b_a) if a is not None]
            if not valid: continue
            b_p, b_ip, b_a = zip(*valid)

            try:
                b_img = [Image.open(p).convert('RGB') for p in b_ip]
            except:
                continue

            inp = processor(text=list(b_p), images=b_img, return_tensors="pt", padding=True)
            inp = {k: v.to(device) for k, v in inp.items()}

            out  = model(inp['input_ids'], inp['attention_mask'], inp['pixel_values'])
            tgt  = torch.tensor([answer_token_ids[a] for a in b_a], device=device)
            loss = torch.nn.functional.cross_entropy(out.logits[:, -1, :], tgt)

            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            total_loss += loss.item(); n += 1

        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {total_loss/max(n,1):.4f}")

    val_preds  = get_prediction(model, processor, batch_size, device, val_data)
    test_preds = get_prediction(model, processor, batch_size, device, test_data)
    tag = name.replace(" ", "_").replace("-", "").replace("/", "")
    write_answers_to_file('vqa_val_student_with_image_ids.jsonl',  val_preds,  answer_to_ids, tag=tag)
    write_answers_to_file('vqa_test_student_with_image_ids.jsonl', test_preds, answer_to_ids, tag=tag)

    acc = evaluation('vqa_val_teacher.jsonl', 'vqa_val_student_final.jsonl')
    print(f"\n→ Val accuracy: {acc*100:.2f}%  (baseline: 50.94%)")

    del model; torch.cuda.empty_cache()
    return acc


# ── Experiment 2: Prompt Engineering — Improved prompt only ──────────
# Intuition: The baseline prompt gives only a terse instruction
# ("Answer with A, B, C, or D only."). A more explicit prompt that
# directs the model to inspect the image and uses a stronger answer cue
# ("The answer is:") helps the model attend to visual information and
# narrows the output distribution towards a single letter.
torch.cuda.empty_cache()
acc2 = run_experiment(
    "Exp2 - Better Prompt only",
    num_epochs=3,
    use_enhanced_projector=False,
    use_v2_prompt=True,
    use_warmup=False,
    peak_lr=1e-4
)

EXPERIMENT: Exp2 - Better Prompt only
  enhanced_proj=False | v2_prompt=True | warmup=False | lr=0.0001


Trainable params: 597,614,592
Train 56055 | Val 3300 | Test 5538
Epoch 1/3 | Loss: 1.4252
Epoch 2/3 | Loss: 1.1134
Epoch 3/3 | Loss: 0.9198
Saved: vqa_val_student_final_Exp2__Better_Prompt_only.jsonl
Saved: vqa_test_student_final_Exp2__Better_Prompt_only.jsonl
→ Val accuracy: 54.36%  (baseline: 50.94%)


> **Note on Experiment 2:** this cell writes its predictions to `..._final_Exp2__Better_Prompt_only.jsonl` but evaluates the untagged `vqa_val_student_final.jsonl`, which still holds Experiment 1's predictions. The 54.36% printed above is therefore Experiment 1's score, not a measurement of the prompt change. The bug is fixed in the Experiment 3 cell below.

In [ ]:
import math

class EnhancedProjector(nn.Module):
    def __init__(self, vision_hidden_size, text_hidden_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(vision_hidden_size, text_hidden_size),
            nn.LayerNorm(text_hidden_size), nn.GELU(),
            nn.Linear(text_hidden_size, text_hidden_size),
            nn.LayerNorm(text_hidden_size), nn.GELU(),
            nn.Linear(text_hidden_size, text_hidden_size),
        )
    def forward(self, x):
        return self.net(x)


def load_image_vqa_data_v2(path, answer_to_ids):
    image_paths, prompts, answers = [], [], []
    for line in open(path, 'r'):
        jdata  = json.loads(line)
        opts   = jdata['options']
        prompt = (
            "<image>\n"
            "Inspect the image carefully and answer the question. "
            "Reply with only A, B, C, or D.\n"
            f"Question: {jdata['question']}\n---\n"
            f"A. {opts[0]}\nB. {opts[1]}\nC. {opts[2]}\nD. {opts[3]}\n---\n"
            "The answer is:"
        )
        image_paths.append(jdata['image'])
        prompts.append(prompt)
        answers.append(answer_to_ids.get(jdata['answer'], None))
    return {"image_paths": image_paths, "prompts": prompts, "answers": answers}


def get_warmup_cosine_scheduler(optimizer, warmup_steps, total_steps):
    """Linear warmup for warmup_steps, then cosine decay to 1% of peak LR."""
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.01, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def build_base_model(tokenizer, image_token_id, use_enhanced_projector=False):
    clip_full    = CLIPModel.from_pretrained(CLIP_MODEL_ID)
    vision_model = clip_full.vision_model
    del clip_full; torch.cuda.empty_cache()

    qwen_model = AutoModelForCausalLM.from_pretrained(TEXI_MODEL_ID)
    qwen_model.resize_token_embeddings(len(tokenizer))

    vision_h = vision_model.config.hidden_size
    text_h   = qwen_model.config.hidden_size

    if use_enhanced_projector:
        projector = EnhancedProjector(vision_h, text_h)
    else:
        import types
        cfg = types.SimpleNamespace(
            vision_config=types.SimpleNamespace(hidden_size=vision_h),
            text_config=types.SimpleNamespace(hidden_size=text_h),
            projector_hidden_act="gelu"
        )
        projector = LlavaMultiModalProjector(cfg)

    return vision_model, projector, qwen_model


def write_answers_to_file(path, predictions, answer_to_ids, tag=""):
    suffix = f"_{tag}" if tag else ""
    output_path = path.replace('with_image_ids', f'final{suffix}')
    ids_to_answer = {v: k for k, v in answer_to_ids.items()}
    with open(output_path, 'w', encoding='utf-8') as f_out:
        for line, pred in zip(open(path, 'r', encoding='utf-8'), predictions):
            jdata = json.loads(line)
            jdata['answer'] = ids_to_answer[pred] if isinstance(pred, int) else pred
            f_out.write(json.dumps(jdata) + '\n')
    print(f"Saved: {output_path}")


def run_experiment(name, num_epochs, use_enhanced_projector, use_v2_prompt, use_warmup, peak_lr=1e-4):
    """Trains and evaluates one experiment. Returns validation accuracy."""
    print(f"\n{'='*55}")
    print(f"EXPERIMENT: {name}")
    print(f"  enhanced_proj={use_enhanced_projector} | v2_prompt={use_v2_prompt} | warmup={use_warmup} | lr={peak_lr}")
    print(f"{'='*55}")

    random.seed(0); torch.manual_seed(0)
    device         = "cuda" if torch.cuda.is_available() else "cpu"
    answer_to_ids  = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
    answer_letters = ['A', 'B', 'C', 'D']

    tokenizer = AutoTokenizer.from_pretrained(TEXI_MODEL_ID)
    tokenizer.add_special_tokens({"additional_special_tokens": ["<image>"]})
    image_token_id      = tokenizer.convert_tokens_to_ids("<image>")
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    vision_model, projector, qwen_model = build_base_model(
        tokenizer, image_token_id, use_enhanced_projector
    )
    model = CustomLLaVA(vision_model, projector, qwen_model, image_token_id).to(device)
    for p in model.vision_model.parameters():
        p.requires_grad = False

    print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    image_processor = CLIPImageProcessor.from_pretrained(CLIP_MODEL_ID)
    processor       = SimpleProcessor(tokenizer, image_processor)

    loader     = load_image_vqa_data_v2 if use_v2_prompt else load_image_vqa_data
    train_data = loader('vqa_train_student.jsonl',                 answer_to_ids)
    val_data   = loader('vqa_val_student_with_image_ids.jsonl',  answer_to_ids)
    test_data  = loader('vqa_test_student_with_image_ids.jsonl', answer_to_ids)
    print(f"Train {len(train_data['prompts'])} | Val {len(val_data['prompts'])} | Test {len(test_data['prompts'])}")

    batch_size       = 8
    answer_token_ids = [tokenizer.encode(l, add_special_tokens=False)[0] for l in answer_letters]
    optimizer        = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=peak_lr, weight_decay=0.01
    )
    total_steps = num_epochs * (len(train_data['prompts']) // batch_size)

    if use_warmup:
        scheduler = get_warmup_cosine_scheduler(optimizer, int(total_steps * 0.05), total_steps)
    else:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-6)

    for epoch in range(num_epochs):
        model.train()
        total_loss, n = 0.0, 0
        indices = list(range(len(train_data['prompts']))); random.shuffle(indices)

        for i in range(0, len(indices), batch_size):
            idx  = indices[i:i+batch_size]
            b_p  = [train_data['prompts'][j]     for j in idx]
            b_ip = [train_data['image_paths'][j] for j in idx]
            b_a  = [train_data['answers'][j]     for j in idx]

            valid = [(p,ip,a) for p,ip,a in zip(b_p,b_ip,b_a) if a is not None]
            if not valid: continue
            b_p, b_ip, b_a = zip(*valid)

            try:
                b_img = [Image.open(p).convert('RGB') for p in b_ip]
            except:
                continue

            inp = processor(text=list(b_p), images=b_img, return_tensors="pt", padding=True)
            inp = {k: v.to(device) for k, v in inp.items()}
            out  = model(inp['input_ids'], inp['attention_mask'], inp['pixel_values'])
            tgt  = torch.tensor([answer_token_ids[a] for a in b_a], device=device)
            loss = torch.nn.functional.cross_entropy(out.logits[:, -1, :], tgt)

            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            total_loss += loss.item(); n += 1

        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {total_loss/max(n,1):.4f}")

    tag = name.replace(" ", "_").replace("-", "").replace("/", "")
    val_preds  = get_prediction(model, processor, batch_size, device, val_data)
    test_preds = get_prediction(model, processor, batch_size, device, test_data)
    write_answers_to_file('vqa_val_student_with_image_ids.jsonl',  val_preds,  answer_to_ids, tag=tag)
    write_answers_to_file('vqa_test_student_with_image_ids.jsonl', test_preds, answer_to_ids, tag=tag)

    acc = evaluation('vqa_val_teacher.jsonl', f'vqa_val_student_final_{tag}.jsonl')
    print(f"\n→ Val accuracy: {acc*100:.2f}%  (baseline: 50.94%)")

    del model; torch.cuda.empty_cache()
    return acc


# ── Experiment 3: Hyperparameter Tuning — Warmup LR only ─────────────
# Intuition: Starting with a high learning rate immediately destabilises
# the randomly-initialised projector in early batches. A linear warmup
# phase keeps updates small until the projector finds a reasonable
# starting point; cosine decay then reduces the LR smoothly so the
# model converges without oscillating near the minimum.
torch.cuda.empty_cache()
acc3 = run_experiment(
    "Exp3 Warmup LR only",
    num_epochs=3,
    use_enhanced_projector=False,
    use_v2_prompt=False,
    use_warmup=True,
    peak_lr=2e-4
)

print(f"\n{'='*40}")
print(f"Baseline (Task 2)    : 50.94%")
print(f"Exp1 - Enhanced Proj : {acc1*100:.2f}%")
print(f"Exp2 - Better Prompt : {acc2*100:.2f}%")
print(f"Exp3 - Warmup LR     : {acc3*100:.2f}%")
print(f"{'='*40}")

EXPERIMENT: Exp3 Warmup LR only
  enhanced_proj=False | v2_prompt=False | warmup=True | lr=0.0002


Trainable params: 597,614,592
Train 56055 | Val 3300 | Test 5538
Epoch 1/3 | Loss: 1.4691
Epoch 2/3 | Loss: 1.4044
Epoch 3/3 | Loss: 1.2275
Saved: vqa_val_student_final_Exp3_Warmup_LR_only.jsonl
Saved: vqa_test_student_final_Exp3_Warmup_LR_only.jsonl
→ Val accuracy: 48.64%  (baseline: 50.94%)
Baseline (Task 2)    : 50.94%
Exp1 - Enhanced Proj : 54.36%
Exp2 - Better Prompt : 54.36%
Exp3 - Warmup LR     : 48.64%
